<a href="https://colab.research.google.com/github/shannonnonshan/22110042_IS_Lab2/blob/main/whisper_4_chatland.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai-whisper fastapi uvicorn nest-asyncio pyngrok python-multipart


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
!ngrok config add-authtoken 33KiyrBXZZIEFi0r8Qy097QcfcI_7nW6g5fmhZcRhxGAUaNix


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
import whisper
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import threading
from pydub import AudioSegment
import os

# Bắt buộc trên Colab
nest_asyncio.apply()

app = FastAPI()

# CORS (để nhận request từ localhost hoặc domain khác)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # hoặc ["http://localhost:3000"] để an toàn hơn
    allow_methods=["*"],
    allow_headers=["*"],
)

# Load model một lần khi khởi động
model = whisper.load_model("tiny")  # tiny/base/small

@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...)):
    try:
        # Lưu tạm file lên Colab
        input_path = f"/content/{file.filename}"
        with open(input_path, "wb") as f:
            f.write(await file.read())

        # Convert sang WAV nếu cần
        ext = file.filename.split(".")[-1].lower()
        wav_path = input_path
        if ext not in ["wav", "mp3"]:  # nếu webm/ogg
            wav_path = f"/content/{file.filename}.wav"
            audio = AudioSegment.from_file(input_path)
            audio.export(wav_path, format="wav")

        # Transcribe
        result = model.transcribe(wav_path, language="vi")

        # Cleanup
        os.remove(input_path)
        if wav_path != input_path:
            os.remove(wav_path)

        return {"text": result["text"]}
    except Exception as e:
        return {"error": str(e)}

# Kill any running ngrok processes
!pkill ngrok
ngrok.kill()

# Tạo public URL ngrok
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

# Chạy uvicorn trong thread
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run)
thread.start()

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
100%|██████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 101MiB/s]


Public URL: NgrokTunnel: "https://unnullified-outragedly-junior.ngrok-free.dev" -> "http://localhost:8000"
